# 📊 ShieldPay — 01: Exploratory Data Analysis

Understand the dataset structure, class imbalance, feature distributions, and correlations before any modelling.

In [ ]:
# Install required packages (run once)
# !pip install pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.facecolor'] = '#0c1118'
plt.rcParams['axes.facecolor']   = '#131b24'
plt.rcParams['axes.edgecolor']   = '#1e2d3d'
plt.rcParams['text.color']       = '#e8edf2'
plt.rcParams['axes.labelcolor']  = '#e8edf2'
plt.rcParams['xtick.color']      = '#6b7a8d'
plt.rcParams['ytick.color']      = '#6b7a8d'
plt.rcParams['grid.color']       = '#1e2d3d'
plt.rcParams['font.size']        = 11

print('✅ Libraries loaded')

## 1. Load Dataset

> **Download:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
> Place `creditcard.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Basic Info

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Basic Statistics ===')
df.describe().T

## 3. Class Distribution (Imbalance Check)

In [ ]:
counts = df['Class'].value_counts()
fraud_pct = counts[1] / len(df) * 100

print(f'Legitimate transactions : {counts[0]:>7,}  ({100-fraud_pct:.4f}%)')
print(f'Fraudulent transactions : {counts[1]:>7,}  ({fraud_pct:.4f}%)')
print(f'Imbalance ratio        : 1 fraud per {int(counts[0]/counts[1])} legitimate')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], counts.values,
            color=['#0099ff', '#ff4757'], width=0.5, edgecolor='none')
axes[0].set_title('Class Distribution (Count)', pad=12, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', color='#e8edf2', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            colors=['#0099ff', '#ff4757'], autopct='%1.4f%%',
            startangle=90, wedgeprops={'edgecolor': '#0c1118', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)', pad=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eda_class_distribution.png')

## 4. Amount & Time Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount distribution by class
for cls, color, label in [(0,'#0099ff','Legitimate'), (1,'#ff4757','Fraud')]:
    axes[0,0].hist(df[df['Class']==cls]['Amount'],
                   bins=80, alpha=0.7, color=color, label=label, edgecolor='none')
axes[0,0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0,0].set_xlabel('Amount (€)')
axes[0,0].set_ylabel('Count')
axes[0,0].legend()
axes[0,0].set_yscale('log')

# Amount boxplot
fraud_amounts = df[df['Class']==1]['Amount']
legit_amounts = df[df['Class']==0]['Amount']
axes[0,1].boxplot([legit_amounts, fraud_amounts],
                   labels=['Legitimate', 'Fraud'],
                   patch_artist=True,
                   boxprops={'facecolor':'#1e2d3d', 'color':'#6b7a8d'},
                   medianprops={'color':'#00d4aa', 'linewidth':2},
                   whiskerprops={'color':'#6b7a8d'},
                   capprops={'color':'#6b7a8d'},
                   flierprops={'marker':'o','markerfacecolor':'#ff4757','alpha':0.3,'markersize':3})
axes[0,1].set_title('Amount Boxplot by Class', fontweight='bold')
axes[0,1].set_ylabel('Amount (€)')

# Time distribution
axes[1,0].hist(df[df['Class']==0]['Time']/3600, bins=48, alpha=0.7,
               color='#0099ff', label='Legitimate', edgecolor='none')
axes[1,0].hist(df[df['Class']==1]['Time']/3600, bins=48, alpha=0.7,
               color='#ff4757', label='Fraud', edgecolor='none')
axes[1,0].set_title('Transaction Time (Hours)', fontweight='bold')
axes[1,0].set_xlabel('Hours since start')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

# Amount stats table
stats = df.groupby('Class')['Amount'].agg(['mean','median','std','max'])
stats.index = ['Legitimate', 'Fraud']
axes[1,1].axis('off')
table = axes[1,1].table(
    cellText=[[f'€{v:.2f}' for v in row] for row in stats.values],
    rowLabels=stats.index,
    colLabels=['Mean', 'Median', 'Std Dev', 'Max'],
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.3, 2.2)
axes[1,1].set_title('Amount Statistics by Class', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('eda_amount_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eda_amount_time.png')

## 5. PCA Feature Distributions (V1–V28)

In [ ]:
# Mean value of each V feature per class
v_cols = [f'V{i}' for i in range(1, 29)]
means = df.groupby('Class')[v_cols].mean().T
means.columns = ['Legitimate', 'Fraud']

fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(v_cols))
width = 0.4
ax.bar(x - width/2, means['Legitimate'], width, label='Legitimate', color='#0099ff', alpha=0.8)
ax.bar(x + width/2, means['Fraud'],      width, label='Fraud',       color='#ff4757', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(v_cols, rotation=45, fontsize=9)
ax.set_title('Mean Feature Value per Class (V1–V28)', fontweight='bold', fontsize=13)
ax.set_ylabel('Mean Value')
ax.legend()
ax.axhline(0, color='#6b7a8d', linewidth=0.7, linestyle='--')
plt.tight_layout()
plt.savefig('eda_feature_means.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: eda_feature_means.png')

## 6. Correlation Heatmap

In [ ]:
corr = df[v_cols + ['Amount','Time','Class']].corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#ff4757' if v > 0 else '#00d4aa' for v in corr.values]
bars = ax.barh(corr.index, corr.values, color=colors, edgecolor='none')
ax.axvline(0, color='#6b7a8d', linewidth=1)
ax.set_title('Feature Correlation with Fraud (Class)', fontweight='bold', fontsize=13)
ax.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, corr.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8, color='#e8edf2')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

top_pos = corr.nlargest(5)
top_neg = corr.nsmallest(5)
print('🔴 Top positive correlation (fraud indicator):')
print(top_pos.to_string())
print()
print('🟢 Top negative correlation (legit indicator):')
print(top_neg.to_string())

## 7. Key EDA Findings

| Finding | Detail |
|---|---|
| **Severe imbalance** | Only 0.17% fraud — must handle before training |
| **Best fraud indicators** | V17, V14, V12, V10 (strong negative correlation with fraud) |
| **Amount** | Fraud transactions tend to be smaller on average (less suspicious-looking) |
| **Time** | Fraud occurs more at night (low-activity hours) |
| **No missing values** | Dataset is clean — no imputation needed |

➡️ Next: **02_Preprocessing.ipynb**